# W&B results as a LaTeX table

Run all cells to query `drigoni/d4-hparams` for the exact tag `Last`. Keep only
model names with **exactly three runs**, and report their mean and population
standard deviation (`ddof=0`), matching `gather_wandb_results.ipynb`.

Columns: **QM9, GDB13, ZINC**. Rows: **MiDi, MiDi-MMFF2, MiDi-AD2, D4,
D4-MMFF2, D4-AD2**. Validity, uniqueness, and novelty (both mean and standard
deviation) are multiplied by 100; bond distance keeps its original scale.
Missing results are `/`. A metric must exist and be finite in all three runs.
The notebook recognizes `atom_details`, `atom-details`, `atom_types`, optional
`-vN` suffixes, and `zinc250k` as an alias for ZINC. If multiple qualifying run
names map to the same table entry, it raises an error rather than merging them.

**Displayed output:** initialized from the saved `Last` results in the original
notebook, already rounded to five decimal places. It is a saved snapshot;
rerunning all cells fetches current summaries and recomputes from their full
precision. No W&B query was executed to initialize this snapshot.

The table uses your `\nlvalpm` and `\sota` commands. If needed, add these defaults
to your LaTeX preamble along with the required packages:

```latex
\usepackage{booktabs,multirow}
\providecommand{\nlvalpm}[2]{\ensuremath{#1\,{\scriptstyle\pm}\,#2}}
\providecommand{\sota}{state-of-the-art}
```


In [1]:
from pathlib import Path

ENTITY = "drigoni"
PROJECT = "d4-hparams"
RUN_TAG = "Last"

# The table always includes these columns and rows, even when results are missing.
DATASETS = ("qm9", "gdb13", "zinc")
ROWS = (
    ("midi", "base", "MiDi"),
    ("midi", "mmff", "MiDi-MMFF2"),
    ("midi", "atom_details", "MiDi-AD2"),
    ("d4", "base", "D4"),
    ("d4", "mmff", "D4-MMFF2"),
    ("d4", "atom_details", "D4-AD2"),
)
METRICS = (
    ("test/molecular_validity", 100),
    ("test/molecular_uniqueness", 100),
    ("test/molecular_novelty", 100),
    ("test/bond_distance", 1),
)

# Optional export in addition to printing the LaTeX. For example: Path("metrics_table.tex").
OUTPUT_PATH = None


In [2]:
from collections import defaultdict
import math
import re
import statistics
import sys


def parse_model_name(run_name):
    """Map known run names to a dataset/model/representation table cell."""
    normalized = run_name.lower().replace("_", "-")
    match = re.fullmatch(
        r"(qm9|gdb13|zinc|zinc250k)-(midi|d4)"
        r"(?:-(mmff|atom-details|atom-types)(?:-v\d+)?)?",
        normalized,
    )
    if match is None:
        return None
    dataset, model, representation = match.groups()
    dataset = "zinc" if dataset == "zinc250k" else dataset
    representation = {None: "base", "mmff": "mmff",
                      "atom-details": "atom_details", "atom-types": "atom_details"}[representation]
    return dataset, model, representation


def summarize_runs(runs):
    """Use exactly three tagged runs per exact name and population std (ddof=0)."""
    grouped = defaultdict(list)
    for run in runs:
        if parse_model_name(run.name or "") is not None:
            grouped[run.name].append(run)

    results = {}
    for run_name, group in sorted(grouped.items()):
        if len(group) != 3:
            print(f"Skipping {run_name}: expected exactly 3 runs, found {len(group)}.", file=sys.stderr)
            continue
        key = parse_model_name(run_name)
        if key in results:
            raise ValueError(
                f"Several model names map to {key}: {results[key]['run_name']} and {run_name}. "
                f"Keep the tag {RUN_TAG!r} on only the intended three-run group."
            )
        metrics = {}
        for metric, _ in METRICS:
            values = []
            for run in group:
                try:
                    value = float(run.summary[metric])
                except (KeyError, TypeError, ValueError, OverflowError):
                    break
                if not math.isfinite(value):
                    break
                values.append(value)
            if len(values) != 3:
                print(f"Missing/non-finite {metric} in {run_name}; table entry will be /.", file=sys.stderr)
                continue
            metrics[metric] = {"mean": statistics.mean(values), "std": statistics.pstdev(values)}
        results[key] = {"run_name": run_name, "metrics": metrics}
    return results


def gather_results(api=None):
    """Read summaries from W&B, restricted to the exact, case-sensitive tag."""
    if api is None:
        import wandb
        api = wandb.Api()
    runs = api.runs(f"{ENTITY}/{PROJECT}", filters={"tags": {"$in": [RUN_TAG]}})
    return summarize_runs(runs)


def latex_value(metrics, metric, scale):
    values = metrics.get(metric)
    if values is None:
        return "/"
    mean, std = values["mean"] * scale, values["std"] * scale
    return rf"\nlvalpm{{{mean:.5f}}}{{{std:.5f}}}"


def make_latex_table(results):
    """Render six model rows; percentages are scaled exactly once at display."""
    lines = [
        r"\begin{table}[t]",
        r"\setlength{\tabcolsep}{2.pt}",
        r"\small",
        r"\centering",
        r"    \begin{tabular}{@{}lcccccccccccc@{}}",
        r"      \toprule",
        r"      \multirow{2}[1]{*}{Model}",
        r"      & \multicolumn{4}{c}{QM9}",
        r"      & \multicolumn{4}{c}{GDB13}",
        r"      & \multicolumn{4}{c}{ZINC} \\",
        r"      \cmidrule(lr{4pt}){2-5}",
        r"      \cmidrule(lr{4pt}){6-9}",
        r"      \cmidrule(lr{4pt}){10-13}",
        r"      & Val. & Uniq. & Nov. & Dist.",
        r"      & Val. & Uniq. & Nov. & Dist.",
        r"      & Val. & Uniq. & Nov. & Dist.\\",
        r"      \midrule",
        "",
    ]
    for index, (model, representation, label) in enumerate(ROWS):
        if index == 3:
            lines.extend([r"      \midrule", ""])
        lines.append("    " + label)
        cells = []
        for dataset in DATASETS:
            metrics = results.get((dataset, model, representation), {}).get("metrics", {})
            cells.extend(latex_value(metrics, metric, scale) for metric, scale in METRICS)
        for cell_index, cell in enumerate(cells):
            lines.append("      & " + cell + (r" \\" if cell_index == len(cells) - 1 else ""))
        lines.append("")
    lines.extend([
        r"      \bottomrule",
        r"    \end{tabular}",
        r"    \caption{Comparison of D4 with the \sota{} MiDi model.",
        "    Validity, uniqueness, and novelty are reported in percentages,",
        "    while bond distance is reported in its original scale.}",
        r"    \label{tab:metrics}",
        r"\end{table}",
    ])
    return "\n".join(lines)


In [3]:
# Fetch current W&B summaries. This performs read-only API requests.
results = gather_results()
print(f"Loaded {len(results)} dataset/model/representation groups with exactly 3 runs.")


wandb: Loading settings from /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/wandb/settings
wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/u0074/.netrc.


Loaded 12 dataset/model/representation groups with exactly 3 runs.


In [4]:
# Run the preceding cells first to refresh results from W&B.
latex_table = make_latex_table(results)
print(latex_table)

if OUTPUT_PATH is not None:
    output = Path(OUTPUT_PATH)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(latex_table + "\n", encoding="utf-8")


\begin{table}[t]
\setlength{\tabcolsep}{2.pt}
\small
\centering
    \begin{tabular}{@{}lcccccccccccc@{}}
      \toprule
      \multirow{2}[1]{*}{Model}
      & \multicolumn{4}{c}{QM9}
      & \multicolumn{4}{c}{GDB13}
      & \multicolumn{4}{c}{ZINC} \\
      \cmidrule(lr{4pt}){2-5}
      \cmidrule(lr{4pt}){6-9}
      \cmidrule(lr{4pt}){10-13}
      & Val. & Uniq. & Nov. & Dist.
      & Val. & Uniq. & Nov. & Dist.
      & Val. & Uniq. & Nov. & Dist.\\
      \midrule

    MiDi
      & \nlvalpm{99.67667}{0.20822}
      & \nlvalpm{96.05015}{0.21649}
      & \nlvalpm{31.99089}{2.32370}
      & \nlvalpm{0.01231}{0.00988}
      & \nlvalpm{98.89333}{0.40003}
      & \nlvalpm{100.00000}{0.00000}
      & \nlvalpm{99.91236}{0.01262}
      & \nlvalpm{0.01543}{0.00259}
      & /
      & /
      & /
      & / \\

    MiDi-MMFF2
      & \nlvalpm{99.86333}{0.00471}
      & \nlvalpm{96.25487}{0.34617}
      & \nlvalpm{31.08297}{1.53797}
      & \nlvalpm{0.00779}{0.00619}
      & \nlvalpm{99.60333}{0.0